In [ ]:
from sys import path

path.append("..")

import os
from pathlib import Path

os.chdir(Path.cwd().parent)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from src.graph_utils.reading import read_metadata
from src.settings import Settings


In [ ]:
metadata = read_metadata()

In [ ]:
from collections import Counter

def count_elements(row_array):
    if len(row_array) == 1 and row_array[0] == -1:
        return 0
    return len(row_array)

test_data = pd.read_parquet(Settings.processed_datasets_dir / 'table.parquet')
test_data['count'] = test_data['result'].apply(count_elements)

test_data['percentage'] = test_data.apply(lambda x: x['count'] / metadata[x['dataset_name']]['graph_count'], axis=1)

In [ ]:
test_data = test_data.reset_index(drop=True)
test_data

In [ ]:
test_data['result'].iloc[0]

In [ ]:

test_data['features'] = test_data['features'].astype('string').astype('category')
test_data.columns

In [ ]:
test_data.sort_values('count').head(20)

In [ ]:
import os
os.listdir('datasets/raw')

In [ ]:
# data2d = test_data[~test_data['features'].str.contains(' ')]
# data2d = test_data[~test_data['features'].str.contains('k_graph')]
# data2d = test_data[test_data['features'].str.contains(':k_graph2')]
data2d = test_data


# data2d = test_data[~test_data['features'].str.contains(' ') & (test_data['features'].str.contains('jaccard_index') | test_data['features'].str.contains('cn_triangle'))]
# data2d = test_data[~test_data['features'].str.contains(' ') & (test_data['features'].str.contains('scan') | test_data['features'].str.contains('neighborhood_distance'))]


data2d = data2d.pivot(index="features", columns="dataset_name", values="percentage")
data2d = data2d.sort_index()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = 1 - data2d

# mean
mean_score = df.mean(axis=1)

# power mean
def power_mean(df, p, epsilon=1e-12):
    df_safe = df + epsilon
    
    if p == 0:
        return np.exp(np.log(df_safe).mean(axis=1))
    else:
        return (df_safe.pow(p).mean(axis=1)) ** (1 / p)

# różne p (to miało być Twoim celem)
p0_score = power_mean(df, p=0)    # geometric
p_1_score = power_mean(df, p=-1)  # harmonic
p_2_score = power_mean(df, p=-2)    # strong penalty for zeros


# size
sizes = pd.Series({
    dataset: metadata[dataset]['graph_count']
    for dataset in df.columns
})
weights_size = sizes / sizes.sum()

weighted_mean_size = (df * weights_size).sum(axis=1)

# difficulty
dataset_difficulty = df.mean(axis=0)
difficulty_weights = dataset_difficulty / dataset_difficulty.sum()

weighted_mean_difficulty = (df * difficulty_weights).sum(axis=1)

results = pd.DataFrame({
    "mean": mean_score,
    "p=0 (geom)": p0_score,
    "p=-1 (harm)": p_1_score,
    "p=-2": p_2_score,
    "weighted_size": weighted_mean_size,
    "weighted_difficulty": weighted_mean_difficulty
})


In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(20, 35),
    # figsize=(20, 5),

    gridspec_kw={'width_ratios': [7, 1]}
)

sns.heatmap(
    data2d,
    cmap='plasma_r',
    vmin=0, vmax=1,
    annot=np.round(data2d * 100, 1),
    fmt=".0f",
    cbar=False,
    ax=ax1
)

sns.heatmap(
    results,
    cmap='viridis',
    vmin=0, vmax=1,
    annot=np.round(results * 100, 1),
    fmt=".0f",
    cbar=False,
    ax=ax2
)
ax2.set_yticks([])
ax2.set_ylabel("")

plt.tight_layout()

In [ ]:
len(df)

In [ ]:

# \multirow{3}{4em}{Multiple row} & cell2 & cell3 \\ 
# & cell5 & cell6 \\ 
# & cell8 & cell9 \\ 
# \hline
# \end{tabular}
# \end{center}
# '''
# latex_template

In [ ]:
from typing import List
def name_to_latex(name: str) -> List:
    name = name.replace('\n', '')
    
    if name[0] != '[':
        return [name.replace('_', r'\_')]
    for s in ["[", "]", "'"]:
        name = name.replace(s, "")

    name_list: List[str] = name.split(" ")

    for i, part in enumerate(name_list):
        name_list[i] = part.replace('_', ' ')


    ltp = {"ldp degree", "ldp max", "ldp mean", "ldp min", "ldp std"}
    if len(ltp.intersection(set(name_list))) == 5:
        name_list = ["LTP"]
    if len(set(map(lambda x: x+ ' normalized' if 'degree' not in x else x, ltp)).intersection(set(name_list))) == 5:
        name_list = ["LTP normalized"]
    return name_list

def generate_latex_table(df: pd.DataFrame, bigger=True):
    
    latex_intro = rf'''
\begin{{center}}
\setlength{{\tabcolsep}}{{1.5pt}}
\renewcommand{{\arraystretch}}{{1.3}}
\tiny

\begin{{tabular}}{{ {"|l|" + "c|" * len(df.columns)} }} 
\hline
\\ & \multicolumn{{{len(df.columns)}}}{{|c|}}{{Datasets}} \\
\hline
Descriptor & { ' & '.join(map(lambda col: rf"\begin{{sideways}} {', '.join(name_to_latex(col))} \end{{sideways}}" ,df.columns)) } \\
\hline
'''

    latex_outro = rf'''
\hline
\end{{tabular}}
\end{{center}}
'''
    tex = latex_intro
    
    for row in df.iterrows():
        name = row[0]
        
        name = name_to_latex(name)
        new_line = ", ".join(name) + " & "
        
        values = np.round(row[1] , 2)
        best_score = np.argwhere(values == (np.amax(values) if bigger else np.amin(values))).flatten()
        
        def values_to_latex(x):
            i = x[0]
            x = x[1]
            return(
                (r"\cellcolor[HTML]{AAAA00}" if i in best_score else "") + 
                ('0.' if x == 0 else f"{x:.2f}".lstrip('0').replace(".00", "."))
            )
        new_line = new_line + " & ".join(map(
            values_to_latex, enumerate(values.tolist()))) + '\\\\\n'
        
        tex = tex + new_line # + "\\hline\n"
    
    tex = tex + latex_outro
    return tex


    # print(np.round(row[1] * 100, 1).astype(int))

print(generate_latex_table(df.transpose()))  

In [ ]:
for row in df.iterrows():
    name = row[0]
    for s in ["[", "]", "'"]:
        name = name.replace(s, "")
    name = name.replace('_', '\\_')
    name = name.split(" ")

    # TODO: ADD special handles like LTP

    new_line = " ".join(name)
    print(new_line)
    new_line = new_line + " & ".join(map(str, np.round(row[1] * 100, 1).astype(int).tolist())) + '\\'


    np.round(row[1] * 100, 1).astype(int)


    # print(" & ".join(map(str, np.round(row[1] * 100, 1).astype(int).tolist())))

In [ ]:
print((df * 100).astype(int).to_latex())